# Beatloom no Colab ou no Kaggle / Beatloom on Colab or Kaggle

Gera mapas de Beat Saber em lote, numa GPU da nuvem, a partir de uma playlist do YouTube ou de uma pasta de áudios. Não tem a página nem o visualizador 3D: abra os zips depois no Beatloom do seu PC.

Generates Beat Saber maps in bulk on a cloud GPU, from a YouTube playlist or a folder of audio files. There is no page and no 3D viewer here: open the zips later in Beatloom on your PC.

**Google Colab**

1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução** (*Runtime → Change runtime type*), marque uma GPU (*Hardware accelerator*) e clique em **Salvar** (*Save*).
2. Rode as três células de cima para baixo. A primeira pede acesso ao seu Drive. / Run the three cells top to bottom. The first one asks for access to your Drive.
3. Os zips e a biblioteca de padrões ficam em `Meu Drive/Beatloom`. / The zips and the pattern library are kept in `My Drive/Beatloom`.

**Kaggle**

1. Painel da direita, **Session options**: **Accelerator → GPU T4 x2** e **Internet → On** (a conta precisa ter o telefone verificado). / Right panel, **Session options**: **Accelerator → GPU T4 x2** and **Internet → On** (the account needs a verified phone).
2. Rode as três células de cima para baixo. / Run the three cells top to bottom.
3. Os zips aparecem na aba **Output**, em `Beatloom/saida`, e juntos em `beatloom-mapas.zip`. / The zips show up in the **Output** tab, under `Beatloom/saida`, and all together in `beatloom-mapas.zip`.
4. Para não esperar os 20 minutos da biblioteca de novo: baixe os arquivos de `Beatloom/dados` na aba Output, crie um **Dataset privado** com eles e, nas próximas vezes, adicione-o ao notebook em **Add Input**. / To skip the 20-minute library build next time: download the files in `Beatloom/dados` from the Output tab, create a **private Dataset** with them and attach it with **Add Input**.

O Beatloom usa poucos GB de memória de vídeo; qualquer GPU serve. Se a sessão cair, rode tudo de novo: no Colab as músicas já feitas são puladas. / Beatloom needs only a few GB of video memory; any GPU works. If the session dies, run everything again: on Colab finished songs are skipped.

In [ ]:
#@title 1. Preparar (uns 3 minutos) / Setup (about 3 minutes)
import glob, json, os, shutil, sys, time
from pathlib import Path
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA: ligue a GPU nas opcoes do ambiente / NONE: turn the GPU on in the runtime options")
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:  # o que fica em /kaggle/working aparece na aba Output; o repositorio vai para fora dela
    BASE, REPO = "/kaggle/working/Beatloom", "/tmp/beatloom"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE, REPO = "/content/drive/MyDrive/Beatloom", "/content/beatloom"
for p in ("saida", "dados"):  # BASE guarda o que precisa sobreviver: os padroes, os mapas e a lista do que ja foi feito
    os.makedirs(f"{BASE}/{p}", exist_ok=True)
!test -d {REPO} || git clone -q https://github.com/bps2414/beatloom {REPO}
%cd {REPO}
!git pull -q
# o torch ja vem com CUDA; so falta o resto
!pip install -q beat-this==1.1.0 demucs==4.1.0 librosa soundfile yt-dlp
!which npm > /dev/null || apt-get install -y -qq nodejs npm > /dev/null
!cd check && npm ci --no-audit --no-fund --silent
print("pronto / ready")

In [ ]:
#@title 2. Biblioteca de padroes (uns 20 minutos so na primeira vez) / Pattern library (about 20 minutes, first time only)
# deriva de mapas de outros mappers, por isso nao vem no repositorio: e baixada do BeatSaver e guardada com VOCE (Drive, ou um Dataset privado no Kaggle)
ARQS = [f"padroes_{d}.pkl" for d in ("Easy", "Normal", "Hard", "Expert", "ExpertPlus")] + ["tipico.json"]
os.makedirs("dados", exist_ok=True)
guardada = next((p for p in [f"{BASE}/dados"] + [os.path.dirname(a) for a in glob.glob("/kaggle/input/**/padroes_Easy.pkl", recursive=True)] if all(os.path.exists(f"{p}/{a}") for a in ARQS)), None)
if guardada:
    for a in ARQS:
        shutil.copy(f"{guardada}/{a}", "dados")
    print("copiada de / copied from", guardada)
else:
    !python -u tools/download_maps.py 1000
    !python tools/extract_patterns.py
    for a in ARQS:
        shutil.copy(f"dados/{a}", f"{BASE}/dados")
    print("montada e guardada em / built and saved to", f"{BASE}/dados")

In [ ]:
#@title 3. Gerar / Generate
#@markdown Link de playlist ou de video do YouTube, e/ou uma pasta de audios (Colab: `/content/drive/MyDrive/musicas`; Kaggle: `/kaggle/input/nome-do-dataset`). / A YouTube playlist or video link, and/or a folder with audio files.
LINK = "" #@param {type:"string"}
PASTA_DE_AUDIOS = "" #@param {type:"string"}
#@markdown `rapido` nao separa os instrumentos: sai sem notas seguradas e sem paredes. / `rapido` skips instrument separation: no held notes, no walls.
MODO = "completo" #@param ["completo", "rapido"]
DIFICULDADES = "Easy,Normal,Hard,Expert,ExpertPlus" #@param {type:"string"}
DENSIDADE = 1.0 #@param {type:"slider", min:0.8, max:1.3, step:0.05}

sys.path.insert(0, REPO)
import app  # o mesmo caminho do site: app.rodar baixa o audio, chama o gerador e confere o mapa

feitos = Path(BASE, "feitos.txt")
ja = set(feitos.read_text(encoding="utf-8").splitlines()) if feitos.exists() else set()
itens = []
if LINK.strip():
    j = json.loads(app.ytdlp("--skip-download", "--flat-playlist", "--dump-single-json", "--", LINK.strip()))
    itens += [{"link": e["url"], "titulo": e.get("title")} for e in j.get("entries") or [] if e and e.get("url")] if j.get("_type") == "playlist" else [{"link": LINK.strip(), "titulo": j.get("title")}]
if PASTA_DE_AUDIOS.strip():
    itens += [{"arquivo": str(p), "titulo": p.stem} for p in sorted(Path(PASTA_DE_AUDIOS.strip()).iterdir()) if p.suffix.lower() in (".mp3", ".ogg", ".wav", ".m4a", ".flac", ".opus")]
print(f"{len(itens)} musicas / songs")
app.SAIDA.mkdir(exist_ok=True)
for n, it in enumerate(itens, 1):
    chave = f"{MODO} {it.get('link') or it['arquivo']}"
    if chave in ja:
        print(f"[{n}/{len(itens)}] ja feita / already done: {it['titulo']}")
        continue
    trab = {**it, "passos": [], "progresso": None, "densidade": DENSIDADE, "etapa": 3 if MODO == "rapido" else 4,
            "difs": [d for d in app.DIFS if d in DIFICULDADES.replace(" ", "").split(",")] or list(app.DIFS)}
    t0 = time.time()
    app.rodar(trab)
    if trab.get("zip"):  # zip e relatorios saem na hora: se a sessao cair, o que ja foi feito esta salvo
        for p in app.SAIDA.glob(Path(trab["zip"]).stem + ".*"):
            shutil.copy(p, f"{BASE}/saida")
        with feitos.open("a", encoding="utf-8") as f:
            f.write(chave + "\n")
    print(f"[{n}/{len(itens)}] {trab.get('titulo')}: {trab.get('zip') or 'ERRO / ERROR ' + str(trab.get('erro'))} ({time.time() - t0:.0f} s)")
if os.path.isdir("/kaggle/working"):  # um arquivo so para baixar na aba Output
    shutil.make_archive("/kaggle/working/beatloom-mapas", "zip", f"{BASE}/saida")
print(f"fim: os zips estao em / done: the zips are in {BASE}/saida")